### Nama    : Raudhatul Jannah
### NIM     : 2411512016
### Kelas   : B

## 1 RekeningBank dengan RLock

In [ ]:
import threading                          # Mengimpor modul threading untuk mendukung pemograman multi-thread (konkurensi).

class RekeningBank:                       # Mendefinisikan kelas RekeningBank untuk merepresentasikan rekening bank.
    def __init__(self, nama, saldo_awal=0): # Konstruktor kelas untuk inisialisasi objek rekening dengan nama dan saldo awal.
        self.nama = nama                  # Menyimpan nama pemilik rekening.
        self.saldo = saldo_awal           # Menyimpan jumlah saldo awal rekening.
        self._lock = threading.RLock()      # Membuat Reentrant Lock (RLock) untuk mencegah race condition pada instance ini.

    def setor(self, jumlah):              # Metode untuk menyetor atau menambah saldo ke rekening.
        with self._lock:                  # Mengunci RLock agar proses penambahan saldo aman dari akses konkuren.
            self.saldo += jumlah          # Menambahkan jumlah setoran ke saldo saat ini.

    def tarik(self, jumlah):              # Metode untuk menarik atau mengurangi saldo dari rekening.
        with self._lock:                  # Mengunci RLock untuk memastikan pengecekan dan pengurangan saldo aman.
            if jumlah > self.saldo:       # Memeriksa apakah saldo mencukupi untuk penarikan.
                raise ValueError(f"Saldo {self.nama} tidak cukup") # Melempar error jika saldo tidak mencukupi.
            self.saldo -= jumlah          # Mengurangi saldo dengan jumlah penarikan.

    def transfer(self, tujuan, jumlah):   # Metode untuk mentransfer sejumlah dana ke rekening tujuan.
        # Urutkan lock berdasarkan id agar transfer A->B dan B->A tidak saling mengunci (circular wait)
        pertama, kedua = sorted((self, tujuan), key=id) # Mengurutkan objek berdasarkan memori ID guna menghindari deadlock.
        with pertama._lock:               # Mengambil lock objek pertama yang memiliki ID lebih kecil.
            with kedua._lock:             # Mengambil lock objek kedua yang tersisa.
                self.tarik(jumlah)          # Memanggil metode tarik (mengunci ulang _lock secara reentrant).
                tujuan.setor(jumlah)        # Memanggil metode setor (mengunci ulang _lock tujuan secara reentrant).

    def __repr__(self):                   # Metode representasi string resmi untuk objek RekeningBank.
        return f"RekeningBank({self.nama}, saldo={self.saldo})" # Mengembalikan string format informasi rekening dan saldo.

# Uji dasar
a = RekeningBank("A", 1000)               # Membuat instans rekening 'A' dengan saldo awal 1000.
b = RekeningBank("B", 500)                # Membuat instans rekening 'B' dengan saldo awal 500.
a.transfer(b, 300)                        # Melakukan transfer sebesar 300 dari rekening A ke rekening B.
print(a, b)                               # Mencetak status akhir objek rekening a dan b ke konsol.
print("Total saldo:", a.saldo + b.saldo, "(harus tetap 1500)") # Memverifikasi bahwa total saldo gabungan tetap konsisten.

RekeningBank(A, saldo=700) RekeningBank(B, saldo=800)
Total saldo: 1500 (harus tetap 1500)


**Uji konkurensi:** 4 thread melakukan 2.000 transfer bolak-balik (A→B dan B→A) secara bersamaan. Jika locking benar, program selesai dan total saldo tetap.

In [ ]:
import random                               # Mengimpor modul random untuk menghasilkan nominal transfer acak.

a = RekeningBank("A", 100_000)              # Membuat rekening 'A' dengan saldo awal 100.000.
b = RekeningBank("B", 100_000)              # Membuat rekening 'B' dengan saldo awal 100.000.

def bolak_balik(asal, tujuan, n):           # Fungsi untuk melakukan transfer berulang sebanyak 'n' kali.
    for _ in range(n):                      # Melakukan perulangan sebanyak n kali.
        asal.transfer(tujuan, random.randint(1, 10)) # Mentransfer jumlah acak antara 1 s.d. 10 dari 'asal' ke 'tujuan'.

threads = [                                 # Mendefinisikan daftar 4 thread yang berjalan secara bersamaan (konkuren):
    threading.Thread(target=bolak_balik, args=(a, b, 2000)), # Thread 1: Mentransfer dari A ke B sebanyak 2000 kali.
    threading.Thread(target=bolak_balik, args=(b, a, 2000)), # Thread 2: Mentransfer dari B ke A sebanyak 2000 kali.
    threading.Thread(target=bolak_balik, args=(a, b, 2000)), # Thread 3: Mentransfer dari A ke B sebanyak 2000 kali.
    threading.Thread(target=bolak_balik, args=(b, a, 2000)), # Thread 4: Mentransfer dari B ke A sebanyak 2000 kali.
]
for t in threads: t.start()                 # Menjalankan (start) semua thread secara paralel.
for t in threads: t.join()                  # Menunggu (join) hingga seluruh thread selesai dieksekusi.

print(a, b)                                 # Mencetak saldo akhir rekening A dan B setelah semua transaksi selesai.
print("Total saldo:", a.saldo + b.saldo, "-> harus 200000") # Memverifikasi apakah total saldo kedua rekening tetap konsisten (200.000).

RekeningBank(A, saldo=100031) RekeningBank(B, saldo=99969)
Total saldo: 200000 -> harus 200000


**Bukti bahwa `Lock` biasa gagal.** Kelas yang sama dengan `threading.Lock()` dijalankan lewat `subprocess` dengan timeout 3 detik (teknik yang sama seperti di notebook materi).

In [2]:
import subprocess, sys, tempfile, os      # Mengimpor modul untuk menjalankan proses eksternal, manajemen sistem, file sementara, dan jalur OS.

skrip_lock_biasa = """                    # Mendefinisikan string multi-baris yang berisi kode simulasi self-deadlock.
import threading                          # Mengimpor modul threading untuk mendukung pemrogaman multi-thread.

class RekeningBank:                       # Mendefinisikan kelas RekeningBank.
    def __init__(self, saldo):            # Konstruktor kelas untuk inisialisasi saldo awal rekening.
        self.saldo = saldo                # Menyimpan nilai saldo rekening.
        self._lock = threading.Lock()          # Menggunakan Lock BIASA (bukan RLock) yang tidak reentrant.

    def setor(self, j):                   # Metode untuk menyetor atau menambah saldo.
        with self._lock:                  # Mengunci Lock biasa saat menambah saldo.
            self.saldo += j               # Menambahkan jumlah setoran ke saldo.

    def tarik(self, j):                   # Metode untuk menarik atau mengurangi saldo.
        with self._lock:                  # Mengunci Lock biasa saat mengurangi saldo.
            self.saldo -= j               # Mengurangi saldo dengan jumlah penarikan.

    def transfer(self, tujuan, j):        # Metode untuk melakukan transfer dana ke rekening tujuan.
        with self._lock:                       # Mengunci _lock milik 'self' untuk pertama kalinya.
            print('transfer: sudah pegang lock, memanggil self.tarik()...', flush=True) # Mencetak pesan status secara instan ke konsol.
            self.tarik(j)                      # Memanggil tarik() yang mencoba mengunci Lock YANG SAMA -> terjadi self-deadlock (macet).
            tujuan.setor(j)                    # Baris ini tidak akan pernah tercapai karena program sudah macet di atas.

a, b = RekeningBank(100), RekeningBank(0) # Membuat objek rekening 'a' (saldo 100) dan 'b' (saldo 0).
t = threading.Thread(target=a.transfer, args=(b, 10)) # Membuat thread untuk menjalankan proses transfer dari a ke b.
t.start(); t.join()                       # Menjalankan thread dan menunggu prosesnya selesai.
print('selesai')                          # Mencetak teks selesai (tidak akan tercapai jika terjadi deadlock).
"""                                       # Penutup string multi-baris skrip uji.

path = os.path.join(tempfile.gettempdir(), "rekening_lock_biasa.py") # Menentukan path file sementara untuk menyimpan skrip.
with open(path, "w") as f:                 # Membuka file sementara dalam mode tulis.
    f.write(skrip_lock_biasa)             # Menulis string kode skrip ke dalam file tersebut.

try:                                      # Memulai blok penanganan error untuk memantau eksekusi proses.
    subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=3) # Menjalankan file skrip via subprocess dengan batas waktu 3 detik.
    print("Selesai (tidak diharapkan)")   # Dicetak jika proses selesai tanpa timeout (yang sebenarnya tidak diinginkan).
except subprocess.TimeoutExpired as e:    # Menangkap error ketika proses macet dan melebihi batas waktu 3 detik.
    print("TIMEOUT setelah 3 detik -- SELF-DEADLOCK TERKONFIRMASI!") # Menampilkan konfirmasi bahwa self-deadlock berhasil terdeteksi.
    print("Output sebelum macet:", e.stdout if e.stdout else "(kosong)") # Menampilkan output terminal sebelum program mengalami macet total.

TIMEOUT setelah 3 detik -- SELF-DEADLOCK TERKONFIRMASI!
Output sebelum macet: transfer: sudah pegang lock, memanggil self.tarik()...



Method `transfer()` sudah memegang `self._lock` untuk membuat operasi transfer atomik. Di dalamnya ia memanggil `self.tarik()`, yang mencoba meng-*acquire* **lock yang sama** milik objek yang sama. Pada `Lock` biasa, thread yang sama tidak boleh meng-acquire lock yang sudah dipegangnya. Thread akan menunggu dirinya sendiri melepas lock, dan itu tidak akan pernah terjadi (**self-deadlock**, terbukti pada demo di atas). Pada `RLock`, thread pemilik lock boleh meng-acquire lagi. Lock menyimpan penghitung acquire dan baru benar-benar dilepas ketika jumlah `release()` sama dengan jumlah `acquire()`. Maka `transfer()` → `tarik()` serta `transfer()` → `tujuan.setor()` (yang lock-nya sudah dipegang `transfer`) berjalan normal, sementara thread lain tetap terblokir sampai seluruh transfer selesai.
Alternatif tanpa `RLock` adalah memisahkan method `_tarik_tanpa_lock()` agar tidak perlu reentrant. Namun `RLock` lebih sederhana dan aman karena setiap method publik tetap bisa dipanggil sendiri maupun dari method lain.

## 2 Deadlock dengan 3 Lock dan 3 Thread

In [ ]:
import subprocess, sys, tempfile, os      # Mengimpor modul untuk menjalankan proses eksternal, manajemen sistem, file sementara, dan OS.

skrip_deadlock_3 = """                    # Mendefinisikan string multi-baris berisi simulasi circular deadlock dengan 3 lock.
import threading, time                    # Mengimpor modul threading dan time di dalam skrip anak.

lock_a = threading.Lock()                 # Membuat objek Lock A.
lock_b = threading.Lock()                 # Membuat objek Lock B.
lock_c = threading.Lock()                 # Membuat objek Lock C.

def thread1():                            # Mendefinisikan fungsi untuk thread pertama.
    with lock_a:                          # Mengunci Lock A.
        print('Thread1 pegang Lock-A, mau ambil Lock-B...', flush=True) # Menampilkan status pegang A dan ingin B.
        time.sleep(0.3)                   # Menunggu sejenak agar thread lain sempat mengunci lock mereka.
        with lock_b:                      # Mencoba mengunci Lock B (bisa memicu deadlock jika dipegang thread 2).
            print('Thread1 dapat A dan B', flush=True) # Pesan sukses jika berhasil mendapatkan A dan B.

def thread2():                            # Mendefinisikan fungsi untuk thread kedua.
    with lock_b:                          # Mengunci Lock B.
        print('Thread2 pegang Lock-B, mau ambil Lock-C...', flush=True) # Menampilkan status pegang B dan ingin C.
        time.sleep(0.3)                   # Menunggu sejenak.
        with lock_c:                      # Mencoba mengunci Lock C.
            print('Thread2 dapat B dan C', flush=True) # Pesan sukses jika berhasil mendapatkan B dan C.

def thread3():                            # Mendefinisikan fungsi untuk thread ketiga.
    with lock_c:                          # Mengunci Lock C.
        print('Thread3 pegang Lock-C, mau ambil Lock-A...', flush=True) # Menampilkan status pegang C dan ingin A.
        time.sleep(0.3)                   # Menunggu sejenak.
        with lock_a:                      # Mencoba mengunci Lock A.
            print('Thread3 dapat C dan A', flush=True) # Pesan sukses jika berhasil mendapatkan C dan A.

ts = [threading.Thread(target=f) for f in (thread1, thread2, thread3)] # Membuat list berisi 3 thread dari fungsi-fungsi di atas.
for t in ts: t.start()                    # Menjalankan semua thread secara bersamaan.
for t in ts: t.join()                     # Menunggu semua thread selesai (akan macet total di sini karena deadlock).
print('SELESAI - tidak ada deadlock')     # Pesan akhir (tidak akan pernah tercapai).
"""                                       # Penutup string multi-baris skrip uji.

path3 = os.path.join(tempfile.gettempdir(), "demo_deadlock_3.py") # Menentukan path file sementara untuk menyimpan skrip.
with open(path3, "w") as f:                 # Membuka file sementara dalam mode tulis.
    f.write(skrip_deadlock_3)             # Menulis string kode skrip ke dalam file tersebut.

try:                                      # Memulai blok penanganan eksepsi.
    hasil = subprocess.run([sys.executable, path3], capture_output=True, text=True, timeout=3) # Menjalankan file skrip via subprocess dengan batas waktu 3 detik.
    print("Output:", hasil.stdout)        # Mencetak output jika proses selesai tanpa timeout.
except subprocess.TimeoutExpired as e:    # Menangkap error jika proses macet melebihi batas waktu (deadlock).
    print("TIMEOUT setelah 3 detik -- DEADLOCK 3 LOCK TERKONFIRMASI!") # Menampilkan konfirmasi bahwa deadlock berhasil terjadi.
    print("Output sebelum macet total:")  # Judul untuk menampilkan output terminal sebelum macet.
    print(e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or "(kosong)")) # Mencetak output dengan aman baik bertipe bytes maupun string.

TIMEOUT setelah 3 detik -- DEADLOCK 3 LOCK TERKONFIRMASI!
Output sebelum macet total:
Thread1 pegang Lock-A, mau ambil Lock-B...
Thread2 pegang Lock-B, mau ambil Lock-C...
Thread3 pegang Lock-C, mau ambil Lock-A...



**Analisis dengan empat syarat Coffman:**

1. **Mutual exclusion**: tiap `Lock` hanya bisa dipegang satu thread.
2. **Hold and wait**: tiap thread memegang satu lock sambil menunggu lock berikutnya.
3. **No preemption**: tidak ada cara merebut paksa lock dari thread lain.
4. **Circular wait**: T1 → T2 → T3 → T1, siklus tertutup.

Semua thread kini mengambil lock dengan urutan global yang sama, **A → B → C**. Thread3 yang tadinya mengambil `C` lalu `A` sekarang mengambil `A` dulu baru `C`. Dengan begitu siklus tidak mungkin terbentuk, sehingga syarat Coffman #4 (circular wait) hilang.

In [ ]:
skrip_fix_3 = """                         # Mendefinisikan string multi-baris berisi perbaikan deadlock untuk 3 lock.
import threading, time                    # Mengimpor modul threading dan time untuk thread serta jeda waktu.

lock_a = threading.Lock()                 # Membuat objek Lock A.
lock_b = threading.Lock()                 # Membuat objek Lock B.
lock_c = threading.Lock()                 # Membuat objek Lock C.

# ATURAN: urutan global lock adalah A < B < C. Selalu ambil dari yang terkecil. # Komentar aturan hierarki penguncian (lock ordering).

def thread1():                      # butuh A dan B # Mendefinisikan fungsi thread 1 yang memerlukan lock A dan B.
    with lock_a:                          # Mengunci Lock A terlebih dahulu.
        print('Thread1 pegang Lock-A, mau ambil Lock-B...', flush=True) # Menampilkan pesan status pegang A, ingin B.
        time.sleep(0.3)                   # Jeda waktu singkat untuk memberi kesempatan thread lain berjalan.
        with lock_b:                      # Mengunci Lock B.
            print('Thread1 dapat A dan B', flush=True) # Pesan sukses saat berhasil mendapatkan Lock A dan B.

def thread2():                      # butuh B dan C # Mendefinisikan fungsi thread 2 yang memerlukan lock B dan C.
    with lock_b:                          # Mengunci Lock B terlebih dahulu.
        print('Thread2 pegang Lock-B, mau ambil Lock-C...', flush=True) # Menampilkan pesan status pegang B, ingin C.
        time.sleep(0.3)                   # Jeda waktu singkat.
        with lock_c:                      # Mengunci Lock C.
            print('Thread2 dapat B dan C', flush=True) # Pesan sukses saat berhasil mendapatkan Lock B dan C.

def thread3():                      # butuh C dan A -> diurutkan menjadi A lalu C # Mendefinisikan fungsi thread 3 yang diubah urutannya.
    with lock_a:                    # PERBAIKAN: A dulu # Mengunci Lock A terlebih dahulu (sesuai hierarki terkecil).
        print('Thread3 pegang Lock-A, mau ambil Lock-C...', flush=True) # Menampilkan pesan status pegang A, ingin C.
        time.sleep(0.3)                   # Jeda waktu singkat.
        with lock_c:                      # Mengunci Lock C setelah Lock A didapatkan.
            print('Thread3 dapat A dan C', flush=True) # Pesan sukses saat berhasil mendapatkan Lock A dan C.

ts = [threading.Thread(target=f) for f in (thread1, thread2, thread3)] # Membuat daftar 3 thread dengan target fungsi masing-masing.
for t in ts: t.start()                    # Menjalankan semua thread secara bersamaan.
for t in ts: t.join()                     # Menunggu seluruh thread selesai dieksekusi tanpa mengalami deadlock.
print('SELESAI - tidak ada deadlock')     # Menampilkan pesan bahwa program sukses berjalan hingga akhir.
"""                                       # Penutup string multi-baris skrip perbaikan.

path3f = os.path.join(tempfile.gettempdir(), "demo_deadlock_3_fixed.py") # Menentukan path file sementara untuk skrip perbaikan.
with open(path3f, "w") as f:                 # Membuka file sementara dalam mode tulis.
    f.write(skrip_fix_3)                  # Menulis isi string skrip perbaikan ke dalam file tersebut.

hasil = subprocess.run([sys.executable, path3f], capture_output=True, text=True, timeout=5) # Menjalankan file skrip via subprocess dengan batas waktu 5 detik.
print("Output:")                          # Mencetak label header output ke konsol.
print(hasil.stdout)                       # Menampilkan hasil stdout dari eksekusi skrip perbaikan.
print("Return code:", hasil.returncode, "-- 0 berarti BERHASIL tanpa timeout/error") # Menampilkan kode status keluar (0 menandakan sukses tanpa deadlock).

Output:
Thread1 pegang Lock-A, mau ambil Lock-B...
Thread2 pegang Lock-B, mau ambil Lock-C...
Thread2 dapat B dan C
Thread1 dapat A dan B
Thread3 pegang Lock-A, mau ambil Lock-C...
Thread3 dapat A dan C
SELESAI - tidak ada deadlock

Return code: 0 -- 0 berarti BERHASIL tanpa timeout/error


**Mengapa perbaikan ini bekerja:** jika semua thread mengambil lock dalam urutan global yang sama, thread yang memegang lock "terbesar" tidak pernah menunggu lock yang lebih kecil. Rantai tunggu selalu menuju lock berindeks lebih besar, sehingga tidak bisa kembali ke awal dan membentuk lingkaran. Satu dari empat syarat Coffman sudah cukup dihilangkan untuk mencegah deadlock.

## 3 Bounded Buffer dengan Kapasitas Berbeda

In [ ]:
import threading, time, random            # Mengimpor modul threading, time, dan random untuk konkurensi, jeda waktu, serta nilai acak.

def jalankan(ukuran, n_prod=2, n_cons=2, item_per_prod=6, cetak=True, seed=42, # Mendefinisikan fungsi simulasi Producer-Consumer beserta parameter default-nya.
             t_prod=(0.02, 0.08), t_cons=(0.02, 0.08)): # Rentang waktu acak untuk proses produksi dan konsumsi.
    random.seed(seed)                     # Mengatur seed random agar hasil acak pada simulasi bersifat deterministik.
    buffer = []                           # Membuat list kosong sebagai buffer penyimpanan item bersama.
    mutex = threading.Lock()              # Membuat Lock untuk mengamankan akses kritis terhadap buffer dan statistik.
    slot_kosong = threading.Semaphore(ukuran) # Semaphore untuk menghitung jumlah slot kosong pada buffer.
    slot_terisi = threading.Semaphore(0)  # Semaphore untuk menghitung jumlah slot terisi pada buffer (mulai dari 0).

    stat = {"prod_blok": 0, "cons_blok": 0, "maks": 0, "urut": []} # Kamus untuk mencatat statistik jalannya simulasi.
    total = n_prod * item_per_prod        # Menghitung total keseluruhan item yang akan diproduksi oleh semua producer.
    jatah_cons = total // n_cons          # Menghitung jumlah item yang harus dikonsumsi oleh setiap consumer.
    awal = time.perf_counter()            # Mencatat waktu awal eksekusi untuk keperluan pencatatan log waktu relatif.

    def catat(teks):                      # Fungsi internal untuk mencetak log kejadian jika opsi cetak bernilai True.
        if cetak:                         # Memeriksa apakah parameter cetak diaktifkan.
            print(f"{time.perf_counter()-awal:5.2f}s  {teks}") # Mencetak waktu relatif beserta teks pesan log.

    def producer(pid):                    # Mendefinisikan fungsi untuk thread Producer.
        for i in range(item_per_prod):    # Melakukan perulangan sebanyak jatah produksi tiap producer.
            item = f"P{pid}-{i}"          # Membuat string nama unik untuk item yang diproduksi.
            time.sleep(random.uniform(*t_prod)) # Mensimulasikan durasi waktu proses produksi item.
            if not slot_kosong.acquire(blocking=False): # Mengecek ketersediaan slot kosong tanpa memblokir thread secara langsung.
                with mutex: stat["prod_blok"] += 1 # Mengunci mutex dan menambah counter statistik producer terblokir.
                catat(f"[Producer-{pid}] BUFFER PENUH, menunggu slot untuk {item}") # Mencetak log bahwa buffer penuh dan producer menunggu.
                slot_kosong.acquire()     # Menunggu secara blokir hingga slot kosong tersedia kembali.
            with mutex:                   # Mengunci mutex untuk mengakses dan memperbarui buffer serta statistik secara aman.
                buffer.append(item)       # Menambahkan item baru ke dalam buffer.
                stat["maks"] = max(stat["maks"], len(buffer)) # Memperbarui catatan ukuran maksimum buffer yang pernah tercapai.
                stat["urut"].append("P")  # Menambahkan tanda "P" ke dalam riwayat urutan eksekusi statistik.
                catat(f"[Producer-{pid}] taruh {item:<5} (buffer: {len(buffer)}/{ukuran})") # Mencetak log sukses menaruh item.
            slot_terisi.release()         # Melepaskan (menaikkan) semaphore slot terisi karena ada item baru.

    def consumer(cid):                    # Mendefinisikan fungsi untuk thread Consumer.
        for _ in range(jatah_cons):       # Melakukan perulangan sebanyak jatah konsumsi tiap consumer.
            if not slot_terisi.acquire(blocking=False): # Mengecek ketersediaan slot terisi tanpa memblokir thread langsung.
                with mutex: stat["cons_blok"] += 1 # Mengunci mutex dan menambah counter statistik consumer terblokir.
                slot_terisi.acquire()     # Menunggu secara blokir hingga ada item di buffer yang bisa diambil.
            with mutex:                   # Mengunci mutex untuk mengambil item dari buffer secara aman.
                item = buffer.pop(0)      # Mengambil item pertama dari buffer.
                stat["urut"].append("C")  # Menambahkan tanda "C" ke dalam riwayat urutan eksekusi statistik.
                catat(f"    [Consumer-{cid}] ambil {item:<5} (buffer: {len(buffer)}/{ukuran})") # Mencetak log sukses mengambil item.
            slot_kosong.release()         # Melepaskan (menaikkan) semaphore slot kosong karena ruang buffer bertambah.
            time.sleep(random.uniform(*t_cons)) # Mensimulasikan durasi waktu proses konsumsi item.

    ts = [threading.Thread(target=producer, args=(i+1,)) for i in range(n_prod)] # Membuat daftar thread untuk seluruh producer.
    ts += [threading.Thread(target=consumer, args=(i+1,)) for i in range(n_cons)] # Menambahkan daftar thread untuk seluruh consumer ke dalam list `ts`.
    for t in ts: t.start()                # Menjalankan seluruh thread producer dan consumer secara bersamaan (paralel).
    for t in ts: t.join()                 # Menunggu hingga seluruh thread selesai mengeksekusi tugasnya sampai tuntas.
    stat["sisa"] = list(buffer)           # Menyimpan daftar sisa item yang tertinggal di dalam buffer ke statistik.
    stat["waktu"] = time.perf_counter() - awal # Menghitung total waktu durasi eksekusi simulasi.
    return stat                           # Mengembalikan kamus hasil statistik simulasi.

hasil_semua = {}                          # Membuat kamus kosong untuk menampung hasil pengujian berbagai ukuran buffer.
for ukuran in (1, 3, 10):                 # Melakukan perulangan pengujian untuk ukuran buffer bernilai 1, 3, dan 10.
    print(f"UKURAN_BUFFER = {ukuran}")    # Menampilkan informasi ukuran buffer yang sedang diuji saat ini ke konsol.
    hasil_semua[ukuran] = jalankan(ukuran) # Menjalankan fungsi simulasi dengan ukuran buffer tersebut dan menyimpan hasilnya.
    print(f"-> Sisa buffer: {hasil_semua[ukuran]['sisa']}\n") # Menampilkan daftar sisa item di buffer setelah simulasi ukuran itu selesai.

UKURAN_BUFFER = 1
 0.02s  [Producer-2] taruh P2-0  (buffer: 1/1)
 0.02s      [Consumer-1] ambil P2-0  (buffer: 0/1)
 0.06s  [Producer-1] taruh P1-0  (buffer: 1/1)
 0.06s      [Consumer-2] ambil P1-0  (buffer: 0/1)
 0.06s  [Producer-2] taruh P2-1  (buffer: 1/1)
 0.06s      [Consumer-1] ambil P2-1  (buffer: 0/1)
 0.12s  [Producer-1] taruh P1-1  (buffer: 1/1)
 0.12s      [Consumer-1] ambil P1-1  (buffer: 0/1)
 0.14s  [Producer-2] taruh P2-2  (buffer: 1/1)
 0.14s      [Consumer-2] ambil P2-2  (buffer: 0/1)
 0.17s  [Producer-1] taruh P1-2  (buffer: 1/1)
 0.17s      [Consumer-1] ambil P1-2  (buffer: 0/1)
 0.17s  [Producer-2] taruh P2-3  (buffer: 1/1)
 0.19s      [Consumer-2] ambil P2-3  (buffer: 0/1)
 0.19s  [Producer-1] taruh P1-3  (buffer: 1/1)
 0.20s      [Consumer-1] ambil P1-3  (buffer: 0/1)
 0.23s  [Producer-1] taruh P1-4  (buffer: 1/1)
 0.23s  [Producer-2] BUFFER PENUH, menunggu slot untuk P2-4
 0.24s      [Consumer-2] ambil P1-4  (buffer: 0/1)
 0.24s  [Producer-2] taruh P2-4  (buffer

In [ ]:
print(f"{'Buffer':>6} | {'Isi maks':>8} | {'Producer menunggu (penuh)':>26} | {'Consumer menunggu (kosong)':>27} | {'Waktu (s)':>9}") # Mencetak header tabel statistik performa dengan format perataan kanan.
print("-" * 90)                           # Mencetak garis pemisah horizontal sepanjang 90 karakter di bawah header.
for u, s in hasil_semua.items():          # Melakukan iterasi untuk mengambil ukuran buffer (u) dan data statistik (s) dari kamus hasil_semua.
    print(f"{u:>6} | {s['maks']:>8} | {s['prod_blok']:>26} | {s['cons_blok']:>27} | {s['waktu']:>9.2f}") # Mencetak baris data statistik sesuai ukuran buffer secara rapi.

print()                                   # Mencetak baris kosong untuk memberikan jarak spasi visual.
print("Pola urutan kejadian (P = producer menaruh, C = consumer mengambil):") # Mencetak judul label keterangan pola urutan interaksi.
for u, s in hasil_semua.items():          # Melakukan iterasi kembali untuk setiap ukuran buffer dan data statistiknya.
    print(f"  buffer={u:>2}: {''.join(s['urut'])}") # Mencetak gabungan string urutan aksi 'P' dan 'C' untuk tiap ukuran buffer.

Buffer | Isi maks |  Producer menunggu (penuh) |  Consumer menunggu (kosong) | Waktu (s)
------------------------------------------------------------------------------------------
     1 |        1 |                          2 |                           6 |      0.38
     3 |        2 |                          0 |                           8 |      0.39
    10 |        2 |                          0 |                           8 |      0.39

Pola urutan kejadian (P = producer menaruh, C = consumer mengambil):
  buffer= 1: PCPCPCPCPCPCPCPCPCPCPCPC
  buffer= 3: PCPPCCPCPCPCPPCCPCPCPCPC
  buffer=10: PCPPCCPCPCPCPPCCPCPCPCPC


### Producer Lebih Cepat dari Consumer
Pada konfigurasi di atas kecepatan producer dan consumer seimbang, sehingga buffer hampir selalu hampir kosong dan ukuran 3 vs 10 nyaris tidak terbedakan. Agar pengaruh ukuran buffer terlihat jelas, eksperimen diulang dengan producer cepat (0,005–0,02 s per item) dan consumer lambat (0,05–0,10 s per item). Log detail tidak dicetak, hanya statistik dan pola urutan.

In [ ]:
hasil_cepat = {}                          # Membuat kamus kosong untuk menampung hasil simulasi dengan kecepatan tinggi.
for ukuran in (1, 3, 10):                 # Melakukan perulangan pengujian untuk ukuran buffer bernilai 1, 3, dan 10.
    hasil_cepat[ukuran] = jalankan(ukuran, cetak=False, t_prod=(0.005, 0.02), t_cons=(0.05, 0.10)) # Menjalankan simulasi tanpa cetak log dengan jeda produksi cepat dan konsumsi lebih lambat.

print(f"{'Buffer':>6} | {'Isi maks':>8} | {'Producer menunggu (penuh)':>26} | {'Consumer menunggu (kosong)':>27} | {'Waktu (s)':>9}") # Mencetak header tabel statistik performa dengan format perataan kanan.
print("-" * 90)                           # Mencetak garis pemisah horizontal sepanjang 90 karakter di bawah header.
for u, s in hasil_cepat.items():          # Melakukan iterasi untuk mengambil ukuran buffer (u) dan data statistik (s) dari kamus hasil_cepat.
    print(f"{u:>6} | {s['maks']:>8} | {s['prod_blok']:>26} | {s['cons_blok']:>27} | {s['waktu']:>9.2f}") # Mencetak baris data statistik ringkas untuk setiap ukuran buffer.

print()                                   # Mencetak baris kosong untuk memberikan jarak spasi visual.
print("Pola urutan kejadian (P = producer menaruh, C = consumer mengambil):") # Mencetak judul label keterangan pola urutan interaksi.
for u, s in hasil_cepat.items():          # Melakukan iterasi kembali untuk setiap ukuran buffer dan data statistiknya.
    print(f"  buffer={u:>2}: {''.join(s['urut'])}") # Mencetak gabungan string urutan aksi 'P' dan 'C' dari hasil simulasi cepat.

Buffer | Isi maks |  Producer menunggu (penuh) |  Consumer menunggu (kosong) | Waktu (s)
------------------------------------------------------------------------------------------
     1 |        1 |                          9 |                           2 |      0.43
     3 |        3 |                          7 |                           2 |      0.48
    10 |       10 |                          0 |                           2 |      0.46

Pola urutan kejadian (P = producer menaruh, C = consumer mengambil):
  buffer= 1: PCPCPCPCPCPCPCPCPCPCPCPC
  buffer= 3: PCPPCPPCPCPCPCPCPCPCPCCC
  buffer=10: PCPCPPPPPPPPPPCCCCCCCCCC


### Analisis: Pengaruh Ukuran Buffer terhadap Pola Interleaving

Angka pada tabel dapat sedikit berbeda setiap kali dijalankan karena penjadwalan thread bersifat non-deterministik. Pola umumnya tetap sama.

**Eksperimen 1: kecepatan producer ≈ consumer**

| Buffer | Pengamatan |
|--------|------------|
| **1** | Pola hampir sepenuhnya **`PCPCPC…`**. Buffer hanya muat satu item, jadi producer harus menunggu consumer mengambil sebelum bisa menaruh lagi. Muncul pesan `BUFFER PENUH` karena producer terpaksa diam. Kedua pihak bekerja bergantian ketat (*lock-step*). |
| **3** | Buffer hanya terisi maksimum 2 dan tidak pernah penuh, sehingga producer tidak pernah menunggu. Interleaving longgar: sesekali muncul `PP` atau `CC` berturut-turut (buffer menampung selisih sesaat). Consumer sering menunggu karena buffer biasanya kosong. |
| **10** | Hampir identik dengan buffer 3: batas atas buffer tidak pernah tercapai, jadi kapasitas ekstra tidak terpakai. |

**Eksperimen 2: producer jauh lebih cepat dari consumer**

| Buffer | Pengamatan |
|--------|------------|
| **1** | Tetap `PCPC…` ketat, dan producer sering menunggu (`BUFFER PENUH` sangat sering). Kecepatan seluruh sistem ditentukan consumer. |
| **3** | Producer sempat "mendahului" sampai maksimum 3 item (muncul `PP`), lalu tetap harus menunggu consumer. |
| **10** | Producer menaruh hampir semua item sekaligus, sampai buffer penuh (isi maks = 10), tanpa pernah menunggu. Polanya berubah menjadi *burst*: deretan `P` panjang, lalu deretan `C` panjang saat consumer menguras buffer. |

**Kesimpulan:**

1. **Buffer kecil memaksa sinkronisasi ketat.** Producer dan consumer saling menunggu di setiap item, interleaving sangat teratur (`PCPC`), dan banyak waktu terbuang untuk menunggu (*blocking*).
2. **Buffer besar memisahkan (decouple) kecepatan kedua sisi.** Producer dan consumer bisa bekerja dalam *burst* dan fluktuasi kecepatan sesaat diserap buffer.
3. **Kapasitas besar hanya berguna jika benar-benar terpakai.** Jika kedua sisi seimbang (Eksperimen 1), buffer 3 dan 10 hampir tidak berbeda.
4. **Ukuran buffer tidak mempercepat total waktu.** Pada kedua eksperimen waktu total hampir sama untuk semua ukuran, karena batas kecepatan ditentukan oleh sisi yang lebih lambat (consumer). Buffer hanya meredam ketidakselarasan, tidak menambah *throughput*.
5. **Keamanan tetap terjaga untuk semua ukuran.** Semua item terkonsumsi dan sisa buffer kosong, tanpa deadlock dan tanpa race condition, berkat kombinasi `slot_kosong`, `slot_terisi`, dan `mutex`.

Catatan teknis: `acquire(blocking=False)` hanya dipakai untuk **mendeteksi** apakah thread harus menunggu (untuk statistik). Jika gagal, thread tetap memanggil `acquire()` biasa (blocking), sehingga logika sinkronisasi tidak berubah.

## 4 Refleksi Singkat

Menurut saya **deadlock** paling berbahaya bagi sistem produksi seperti perbankan atau e-commerce. Thread yang macet terus memegang lock dan koneksi database, sehingga permintaan lain ikut antre dan seluruh layanan bisa lumpuh. Transfer atau pesanan yang sedang berjalan juga berhenti di tengah proses, sehingga data berisiko tidak konsisten. Kerusakannya bersifat permanen sampai ada restart manual.
Namun, yang paling sulit dideteksi saat testing adalah **starvation** (dan **livelock**). Deadlock setidaknya jelas terlihat: program berhenti dan thread dump menunjukkan siklus tunggu. Pada starvation dan livelock, sistem tampak hidup, tidak ada error, CPU sibuk, tetapi sebagian thread nyaris tidak maju. Masalahnya baru muncul pada beban tinggi dan timing tertentu, yang jarang terjadi di lingkungan uji, sehingga sering baru ketahuan di produksi.